# Scrape basic information of cities from Wikipedia

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from lat_lon_parser import parse
from dotenv import load_dotenv, find_dotenv
import os

In [ ]:
def scrape_city_data(city_list: list):
    data_frame_rows = []
    for city in city_list:
        url = "https://en.wikipedia.org/wiki/" + city
        headers = {'User-Agent': 'Chrome/134.0.0.0'}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # country
        info_box_country = soup.find(class_ = 'infobox-label', string=re.compile("country", re.IGNORECASE)) 
        country = info_box_country.find_next().get_text()
        # latitude
        latitude = soup.find(class_ = 'latitude').get_text()
        # longitude
        longitude = soup.find(class_ = 'longitude').get_text()
        # area
        info_box_rows = soup.find_all(class_ = 'infobox-data')
        for td in info_box_rows:
            if 'km' in td.get_text() and td.find('sup'):
                area = td.get_text().split('km')[0].strip()
                break
        
        data_frame_rows.append({
            "city": city,
            "country": country,
            "latitude": round(parse(latitude), 5),
            "longitude": round(parse(longitude), 5),
            "area_km2": round(float(area), 2)
            })
        
        
    return pd.DataFrame(data_frame_rows)

In [17]:
def scrape_city_population_data(city_df: pd.DataFrame):
    data_frame_rows = []
    for city in city_df["city"]:
        url = "https://en.wikipedia.org/wiki/" + city
        headers = {'User-Agent': 'Chrome/134.0.0.0'}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # population
        pop_rows = soup.select("table.infobox tr")
        for i, row in enumerate(pop_rows):
            if "Population" in row.get_text():
                timestamp_population = re.search(r"\d{4}", row.get_text()).group()
                # Try to get population from the current row first
                td = row.find("td")
                if td:
                    population = td.get_text(strip=True)
                # If not in current row, check next row
                elif i + 1 < len(pop_rows):
                    td = pop_rows[i+1].find("td")
                    if td:
                        population = td.get_text(strip=True)
                break
                
        
        data_frame_rows.append({
            "city_id": city_df.loc[city_df["city"] == city, "city_id"].item(),
            "population": int(population.replace(",","")),
            "timestamp_population": f"{timestamp_population}-01-01"
            })
    return pd.DataFrame(data_frame_rows)


In [28]:
list_of_cities = ["Berlin", "Hamburg", "Frankfurt", "Munich", "Cologne", "Stuttgart", "Düsseldorf", "Leipzig", "Dortmund", "Essen"]
city_df = scrape_city_data(list_of_cities)
city_df

,city,country,latitude,longitude,area_km2
0,Berlin,Germany,52.52000,13.40500,891.30
1,Hamburg,Germany,53.55000,10.00000,755.09
2,Frankfurt,Germany,50.11056,8.68222,248.31
3,Munich,Germany,48.13750,11.57500,310.71
4,Cologne,Germany,50.93639,6.95278,405.15
5,Stuttgart,Germany,48.77750,9.18000,207.33
6,Düsseldorf,Germany,51.22556,6.77667,217.41
7,Leipzig,Germany,51.34000,12.37500,297.80
8,Dortmund,Germany,51.51389,7.46528,280.71
9,Essen,Germany,51.45083,7.01306,210.34


# Load data into SQL database

In [ ]:
load_dotenv(find_dotenv())

schema = os.getenv("SQL_schema")
host = os.getenv("SQL_host")
user = os.getenv("SQL_user")
password = os.getenv("SQL_password")
port = os.getenv("SQL_port")

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [32]:
# Step 1: Insert city_df into MySQL
city_df.to_sql('cities',
                  if_exists='delete_rows',
                  con=connection_string,
                  index=False)

10

In [33]:
# Step 2: Retrieve the cities with their auto-generated IDs
city_df_from_sql = pd.read_sql("cities", con=connection_string)
city_df_from_sql

,city_id,city,country,latitude,longitude,area_km2
0,1,Berlin,Germany,52.52000,13.40500,891.30
1,2,Hamburg,Germany,53.55000,10.00000,755.09
2,3,Frankfurt,Germany,50.11056,8.68222,248.31
3,4,Munich,Germany,48.13750,11.57500,310.71
4,5,Cologne,Germany,50.93639,6.95278,405.15
5,6,Stuttgart,Germany,48.77750,9.18000,207.33
6,7,Düsseldorf,Germany,51.22556,6.77667,217.41
7,8,Leipzig,Germany,51.34000,12.37500,297.80
8,9,Dortmund,Germany,51.51389,7.46528,280.71
9,10,Essen,Germany,51.45083,7.01306,210.34


In [34]:
# Step 3: Use city_df_from_sql to scrape population data
population_df = scrape_city_population_data(city_df_from_sql)
population_df

,city_id,population,timestamp_population
0,1,3596999,2022-01-01
1,2,1973896,2024-01-01
2,3,756021,2024-01-01
3,4,1505005,2024-01-01
4,5,1024621,2024-01-01
5,6,612663,2024-01-01
6,7,618685,2024-01-01
7,8,611850,2024-01-01
8,9,603462,2024-01-01
9,10,574682,2024-01-01


In [35]:
# Step 4: Insert population_df into MySQL
population_df.to_sql('city_populations',
                  if_exists='append',
                  con=connection_string,
                  index=False)

10

In [36]:
# Step 5: Retrieve populations_df from MySQL to make sure it is correct
populations_df_from_sql = pd.read_sql("city_populations", con=connection_string)
populations_df_from_sql

,city_id,population,timestamp_population
0,1,3596999,2022-01-01
1,2,1973896,2024-01-01
2,3,756021,2024-01-01
3,4,1505005,2024-01-01
4,5,1024621,2024-01-01
5,6,612663,2024-01-01
6,7,618685,2024-01-01
7,8,611850,2024-01-01
8,9,603462,2024-01-01
9,10,574682,2024-01-01


Wrap everything in functions, so it is only necessary to call one function in the end

In [ ]:
"""
Wikipedia city data scraper.

Scrapes city metadata (country, coordinates, area) and population figures
from Wikipedia infoboxes and pushes the results to a MySQL database.
"""

import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from lat_lon_parser import parse
from dotenv import load_dotenv, find_dotenv
import os

def create_connection_string():
    """Build a SQLAlchemy connection string from environment variables.

    Reads SQL_schema, SQL_host, SQL_user, SQL_password, and SQL_port from
    the nearest .env file and returns a mysql+pymysql:// URL.
    """
    load_dotenv(find_dotenv())
    schema = os.getenv("SQL_schema")
    host = os.getenv("SQL_host")
    user = os.getenv("SQL_user")
    password = os.getenv("SQL_password")
    port = os.getenv("SQL_port")

    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def scrape_city_data(city_list: list):
    """Scrape country, coordinates, and area for each city from Wikipedia.

    Looks for the data inside the Wikipedia infobox:
    - Country:   first infobox-label whose text matches "country" (case-insensitive)
    - Latitude / Longitude: elements with class 'latitude' / 'longitude'
    - Area:      first infobox-data cell that contains 'km' and a <sup> tag
                 (catches the "km²" superscript pattern)

    Args:
        city_list: List of Wikipedia page titles, e.g. ["Berlin", "Paris"].

    Returns:
        DataFrame with columns: city, country, latitude, longitude, area_km2.
    """
    data_frame_rows = []
    for city in city_list:
        url = "https://en.wikipedia.org/wiki/" + city
        headers = {'User-Agent': 'Chrome/134.0.0.0'}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # country
        info_box_country = soup.find(class_ = 'infobox-label', string=re.compile("country", re.IGNORECASE)) 
        country = info_box_country.find_next().get_text()
        # latitude
        latitude = soup.find(class_ = 'latitude').get_text()
        # longitude
        longitude = soup.find(class_ = 'longitude').get_text()
        # area
        info_box_rows = soup.find_all(class_ = 'infobox-data')
        for td in info_box_rows:
            if 'km' in td.get_text() and td.find('sup'):
                area = td.get_text().split('km')[0].strip()
                break
        
        data_frame_rows.append({
            "city": city,
            "country": country,
            "latitude": round(parse(latitude), 5),
            "longitude": round(parse(longitude), 5),
            "area_km2": round(float(area), 2)
            })
        
    return pd.DataFrame(data_frame_rows)

def scrape_city_population_data(city_df: pd.DataFrame):
    """Scrape the most recent population figure for each city from Wikipedia.

    Searches the infobox for a row whose text contains "Population", then
    extracts the year from that row and the numeric value from either the
    same row or the immediately following one (Wikipedia layouts vary).

    Args:
        city_df: DataFrame that must contain 'city' and 'city_id' columns,
                 so the result of retrieve_city_df_from_sql().

    Returns:
        DataFrame with columns: city_id, population, timestamp_population.
        timestamp_population is formatted as "YYYY-01-01".
    """
    data_frame_rows = []
    for city in city_df["city"]:
        url = "https://en.wikipedia.org/wiki/" + city
        headers = {'User-Agent': 'Chrome/134.0.0.0'}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # population
        pop_rows = soup.select("table.infobox tr")
        for i, row in enumerate(pop_rows):
            if "Population" in row.get_text():
                timestamp_population = re.search(r"\d{4}", row.get_text()).group()
                # Try to get population from the current row first
                td = row.find("td")
                if td:
                    population = td.get_text(strip=True)
                # If not in current row, check next row
                elif i + 1 < len(pop_rows):
                    td = pop_rows[i+1].find("td")
                    if td:
                        population = td.get_text(strip=True)
                break
                
        
        data_frame_rows.append({
            "city_id": city_df.loc[city_df["city"] == city, "city_id"].item(),
            "population": int(population.replace(",","")),
            "timestamp_population": f"{timestamp_population}-01-01"
            })
        
    return pd.DataFrame(data_frame_rows)

def send_city_df_to_sql(city_df: pd.DataFrame, connection_string: str):
    """Write the cities DataFrame to the 'cities' table, replacing existing rows.

    Args:
        city_df: DataFrame with columns city, country, latitude, longitude,
                 area_km2.
        connection_string: SQLAlchemy connection URL.
    """
    city_df.to_sql('cities',
                   if_exists='delete_rows',
                   con=connection_string,
                   index=False)

def retrieve_city_df_from_sql(connection_string: str):
    """Load the full 'cities' table from MySQL, including the auto-assigned city_id.

    Args:
        connection_string: SQLAlchemy connection URL.

    Returns:
        DataFrame mirroring the 'cities' table schema.
    """
    return pd.read_sql("cities", con=connection_string)

def send_population_df_to_sql(population_df: pd.DataFrame, connection_string: str):
    """Append population records to the 'city_populations' table.

    Args:
        population_df: DataFrame with columns city_id, population,
                       timestamp_population.
        connection_string: SQLAlchemy connection URL.
    """
    population_df.to_sql('city_populations',
                         if_exists='append',
                         con=connection_string,
                         index=False)

def create_cities_population_tables(cities_list: list):
    """End-to-end pipeline: scrape Wikipedia and populate the MySQL tables.

    Steps:
        1. Scrape city metadata and write to 'cities' (city_id assigned by DB).
        2. Re-read 'cities' to get the auto-assigned city_id values.
        3. Scrape population figures keyed on those IDs.
        4. Append population records to 'city_populations'.

    Args:
        cities_list: List of cities to process.
    """
    connection_string = create_connection_string()
    city_df = scrape_city_data(cities_list)
    send_city_df_to_sql(city_df, connection_string)
    city_df_from_sql = retrieve_city_df_from_sql(connection_string)
    population_df = scrape_city_population_data(city_df_from_sql)
    send_population_df_to_sql(population_df, connection_string)

In [3]:
list_of_cities = ["Berlin", "Hamburg", "Frankfurt", "Munich", "Cologne", "Stuttgart", "Düsseldorf", "Leipzig", "Dortmund", "Essen"]
create_cities_population_tables(list_of_cities)